# Coin flip learning

Consider the following setup: a unknown biased coin, represented by $p = P(\text{HEADS})$, is flipped exactly twice. We know that $p ∈ \{0.3, 0.5, 0.7\}$ and we start with a uniform prior over these three hypotheses.

By symmetry we may assume that the first coin lands HEADS. What is the probability of the second flip landings HEADS as well?

## Classical learning

A classical agent maintains a probability distribution over the three hypotheses. Given this distribution and the value of $p$ in each hypothesis, we can compute an expectation value of each outcome of the second flip.

Upon observing the first coin flip, we use Bayes' rule to update our probability distribution and obtain an updated expectation value.

In [14]:
from enum import Enum
import functools
import operator
import numpy as np
from numpy.typing import NDArray

In [15]:
hypotheses = np.array([0.3, 0.5, 0.7])

# Uniform prior
prior = np.array([1/3, 1/3, 1/3])

# Bayesian update:
# Step 1: p(dist | obs) ~ p(obs | dist) p(dist)
posterior = hypotheses * prior
# Step 2: normalise
posterior /= posterior.sum()

print(f"Prior distribution: [", ",".join("%.3f"%x for x in prior), "]", sep="")
print(f"Probability of first flip landing HEADS:  {(hypotheses*prior).sum():.3f}")
print(f"Posterior distribution: [", ",".join("%.3f"%x for x in posterior), "]", sep="")
print(f"Probability of second flip landing HEADS: {(hypotheses*posterior).sum():.3f}")

Prior distribution: [0.333,0.333,0.333]
Probability of first flip landing HEADS:  0.500
Posterior distribution: [0.200,0.333,0.467]
Probability of second flip landing HEADS: 0.553


## Infrabayesian learning

For the IB implementation we start out by defining the metric space $X$, consisting of all possible histories (four, in this case).

### Helper functions

In [16]:
# Analog to sum()
def prod(factors):
    return functools.reduce(operator.mul, factors, 1)

In [17]:
# Check if the pattern matches a given history
# A pattern can be either a history or a list of histories. In the latter case, the pattern is considered to match if any of any of its elements matches the history.
# If the pattern is a history, each element has to agree with the given history. The value None might be used as a wildcard.
# For example, the pattern (H,None) matches the histories (H,H) and (H,T)
def match(pattern : tuple | list, history : tuple):
    if isinstance(pattern, list):
        # If a list was passed, check each list element
        for p in pattern:
            if match(p, history):
                return True
        return False

    # If a tuple was passed, compare observations element-wise
    for obs_pattern,obs_history in zip(pattern,history):
        if obs_pattern is not None and obs_pattern != obs_history:
            return False
    return True

### Mathematical objects for IB

In [18]:
# Define outcome space
class Coin(Enum):
    H = 0
    T = 1

In [19]:
# Set of possible histories (metric space)
X = [(Coin.H,Coin.H), (Coin.H,Coin.T), (Coin.T,Coin.H), (Coin.T,Coin.T)]

In [20]:
class A_measure:
    """
    An a-measure, characterised by
        a scale factor λ > 0
        a probability measure μ over the history space
        an offset b > 0
    """
    def __init__(self, measure : NDArray[np.float64], scale : float = 1, offset : float = 0):
        assert len(measure) == len(X)
        assert measure.min() >= 0
        self.measure = measure
        self.scale = scale
        self.offset = offset
        self.normalise_measure()

    def expected_value(self, reward_function : callable) -> float:
        """
        Compute the expected value of a given reward function, defined as λ*μ(f) + b
        """
        value = 0
        for i,x in enumerate(X):
            value += self.measure[i] * reward_function(x)
        return self.scale * value + self.offset

    def chop(self, pattern : tuple | list) -> None:
        """
        For updating: set measure to 0 for all events that are inconsistent with the observation
        """
        for i,x in enumerate(X):
            if not match(pattern, x):
                self.measure[i] = 0
        self.normalise_measure()

    def normalise_measure(self) -> None:
        """
        Ensure that mu(1) == 1, put normalisation into scale variable
        This constraint is somewhat arbitrary and one could just merge these variables
        """
        scale = self.measure.sum()
        self.measure /= scale
        self.scale *= scale

    # Helper functions for addition of a-measures, division by a scalar and string formatting
    def __add__(self, other):
        assert isinstance(other, A_measure)
        return A_measure(self.scale*self.measure + other.scale*other.measure, 1, self.offset + other.offset)
    def __itruediv__(self, other : float):
        self.scale /= other
        self.offset /= other
        return self
    def __truediv__(self, other : float):
        return A_measure(self.measure.copy(), self.scale / other, self.offset / other)
    def __repr__(self) -> str:
        return (f"""({self.scale:.3f}[{",".join("%.3f"%x for x in self.measure)}],{self.offset:.3f})""")


In [21]:
# Reward functions map histories (elements of X) to [0,1]
def reward_function_zero(history : list) -> float:
    return 0
def reward_function_one(history : list) -> float:
    return 1
def reward_function_arbitrary(history : list) -> float:
    # Assigns an arbitrary, but constant reward to each history
    return (hash(history) % 1001) / 1000

# Glue operator: func1 *^pattern func2
def glue(func1 : callable, pattern : list, func2 : callable):
    def glued(history):
        return func1(history) if match(pattern,history) else func2(history)
    return glued

In [22]:
class Infradistribution:
    """
    An infradistribution, represented by it extremal minimal points
    I.e. we only keep track of the boundaries of the convex hull of minimal points
    """
    def __init__(self, measures : list[A_measure]):
        self.measures = measures
    
    def expected_value(self, reward_function : callable) -> float:
        """
        Expected value over a given reward function
        Defined as expected value minimum over all minimal points
        """
        return min(measure.expected_value(reward_function) for measure in self.measures)

    def probability(self, reward_function : callable, event : tuple | list) -> float:
        """
        "Probability" of a given event under a given reward function
        This is not a classical probability and it does not satisfy the corresponding properties.
        """
        return (
              self.expected_value(glue(reward_function_one,  event, reward_function))
            - self.expected_value(glue(reward_function_zero, event, reward_function))
        )

    def update(self, reward_function : callable, event : tuple | list) -> None:
        """
        Update all a-measures upon seeing a certain event using a given reward function
        This is Definition 11 from Basic Inframeasure Theory
        """
        expect0 = self.expected_value(glue(reward_function_zero, event, reward_function))
        expect1 = self.expected_value(glue(reward_function_one,  event, reward_function))
        prob = expect1 - expect0
        for measure in self.measures:
            expect_m = measure.expected_value(glue(reward_function_zero, event, reward_function))
            measure.chop(event)
            measure.offset += expect_m - expect0
            measure /= prob

    def __repr__(self) -> str:
        return self.measures.__repr__()

### Implementation of the experiment

Within each of our three hypotheses, we can compute the probabilities of each of each histories. These are the probability measures in our a-measures. Since we use a uniform prior, our initial a-measure is the average over the measures corresponding to the three hypotheses:

In [23]:
def a_measure_from_hypothesis(p : float) -> A_measure:
    p_obs = [p, 1-p]
    mu = np.array([prod(p_obs[obs.value] for obs in x) for x in X])
    return A_measure(mu)

def init_measures():
    global mA,mB,mC,mix
    mA = a_measure_from_hypothesis(0.3)
    mB = a_measure_from_hypothesis(0.5)
    mC = a_measure_from_hypothesis(0.7)
    mix = (mA + mB + mC) / 3

init_measures()
print("a-measures:")
print("m_A =", mA)
print("m_B =", mB)
print("m_C =", mC)
print("mix =", mix)

a-measures:
m_A = (1.000[0.090,0.210,0.210,0.490],0.000)
m_B = (1.000[0.250,0.250,0.250,0.250],0.000)
m_C = (1.000[0.490,0.210,0.210,0.090],0.000)
mix = (1.000[0.277,0.223,0.223,0.277],0.000)


a-measures are shown as as $(λμ,b)$, where $μ$ is a vector with entries $μ(h)$ for all histories $h ∈ X$.

I.e. `m_A = (1.000[0.090,0.210,0.210,0.490],0.000)` means $m_\text{A} = (λμ,b)$ with
$$
    λ = 1 \\
    μ(HH) = 0.090 \\
    μ(HT) = 0.210 \\
    μ(TH) = 0.210 \\
    μ(TT) = 0.490 \\
    b = 0
$$

Next, we define an infradistribution containing only this combined a-measure. Within this distribution, we can compute the probability of the first coin landing HEADS. To compute a probability (or expectation values in general), we need to specify a reward function. In this case, the reward function does not matter, as it cancels in all relevant quantities. To allow the formalism to work, we use a reward function that assigns a pseudo-random (but consistent between calls) reward to each history.

We then use the IB update rule to update our distribution based on actually observing HEADS. Using the updated infradistribution, we can compute the probability of the second flip landing HEADS.

We can re-run the experiment with a different reward function, to verify that it indeed cancels.

In [24]:
event_H  = (Coin.H,None)    # seeing HEADS on the first flip
event_HH = (Coin.H,Coin.H)  # seeing HEADS on both flips

def run_ib_experiment(measures, reward_function):
    infradist = Infradistribution(measures)
    print("Initial infradistribution:", infradist)
    print(f"P(H) = {infradist.probability(reward_function, event_H):.3f}")

    infradist.update(reward_function, event_H)

    print("Updated infradistribution:", infradist)
    print(f"P(HH|H) = {infradist.probability(reward_function, event_HH):.3f}")

init_measures()
run_ib_experiment([mix], reward_function_arbitrary)
#init_measures()
#run_ib_experiment([mix], reward_function_zero)
#init_measures()
#run_ib_experiment([mix], reward_function_one)


Initial infradistribution: [(1.000[0.277,0.223,0.223,0.277],0.000)]
P(H) = 0.500
Updated infradistribution: [(1.000[0.553,0.447,0.000,0.000],0.000)]
P(HH|H) = 0.553


We obtain the same value for $P(HH|H)$ that we got from the classical agent.

## Knightian Uncertainty (?)

For KU, we do not start with a single a-measure (corresponding to uniform mixture over hypotheses), but with multiple a-measures, corresponding to all possible convex combinations of the a-measures corresponding to the three hypotheses. These define a triangle in the convex cone of (s)a-measures. For practical purposes, we only need to keep track of the edges of this triangle (do we?).

As we update, this triangle deforms (?), eventually (??) collapsing into a point (?????).

In [25]:
event_T  = (Coin.T,None)
event_HT = (Coin.H,Coin.T)

In [26]:
reward_functions = [("Zero      ",reward_function_zero),("One       ",reward_function_one),("Arbitrary ",reward_function_arbitrary)]

init_measures()
infradist = Infradistribution([mA, mB, mC])
print("Initial infradistribution:", infradist)
print("Reward func |  P(H) |  P(T)")
for name,func in reward_functions:
    print(f"{name}  | {infradist.probability(func, event_H):.3f} | {infradist.probability(func, event_T):.3f}")

for name,func in reward_functions:
    init_measures()
    infradist = Infradistribution([mA, mB, mC])
    print("\nUpdate on:", name)
    infradist.update(func, event_H)
    print("Updated infradistribution:", infradist)
    print("Reward func | P(HH) | P(HT)")
    for name,func in reward_functions:
        print(f"{name}  | {infradist.probability(func, event_HH):.3f} | {infradist.probability(func, event_HT):.3f}")

Initial infradistribution: [(1.000[0.090,0.210,0.210,0.490],0.000), (1.000[0.250,0.250,0.250,0.250],0.000), (1.000[0.490,0.210,0.210,0.090],0.000)]
Reward func |  P(H) |  P(T)
Zero        | 0.300 | 0.300
One         | 0.700 | 0.700
Arbitrary   | 0.594 | 0.354

Update on: Zero      
Updated infradistribution: [(1.000[0.300,0.700,0.000,0.000],0.000), (1.667[0.500,0.500,0.000,0.000],0.000), (2.333[0.700,0.300,0.000,0.000],0.000)]
Reward func | P(HH) | P(HT)
Zero        | 0.300 | 0.700
One         | 0.300 | 0.700
Arbitrary   | 0.300 | 0.700

Update on: One       
Updated infradistribution: [(0.429[0.300,0.700,0.000,0.000],0.571), (0.714[0.500,0.500,0.000,0.000],0.286), (1.000[0.700,0.300,0.000,0.000],0.000)]
Reward func | P(HH) | P(HT)
Zero        | 0.643 | 0.300
One         | 0.700 | 0.357
Arbitrary   | 0.680 | 0.300

Update on: Arbitrary 
Updated infradistribution: [(0.505[0.300,0.700,0.000,0.000],0.495), (0.841[0.500,0.500,0.000,0.000],0.233), (1.178[0.700,0.300,0.000,0.000],0.000)]
Rew

The "probabilities" are no longer proper probabilities, i.e. they do not add up to one.

The updated infradistribution and probabilities depend on the reward function. This is a general feature of infradistributions containing multiple a-measures, as expectation values are no longer linear.

## Further notes
Things to generalise:

- Longer histories: So far, we need to model $2^n$ histories for $n$ coin flips. Can we reduce the computational complexity? (This would be nice to have, but not crucial for a proof-of-concept implementation. Maybe come back to this later)
- More hypotheses: in KU case, how does computational complexity scale with number of hypotheses? (Somewhat low priority, as above)
- Actually understand how KU behaves (high priority)
- Act based learning information (build an agent!)